Inject Insider Threat Scenarios

In [1]:
import pandas as pd
import numpy as np
import random

random.seed(42)
np.random.seed(42)

print("Libraries loaded successfully.")

Libraries loaded successfully.


2. Loading DBE_Employee_Behaviour.csv

In [4]:
activities = pd.read_csv("DBE_Employee_Behaviour.csv",
                        low_memory=False)

print("Dataset loaded successfully.")


Dataset loaded successfully.


In [6]:
activities = pd.read_csv(
    "DBE_Employee_Behaviour.csv",
    low_memory=False,
    dtype={
        "filename": str,
        "url": str,
        "to_email": str,
        "from_email": str
    }
)

print("Dataset loaded successfully.")

Dataset loaded successfully.


In [8]:
activities.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 21 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   timestamp           50000 non-null  object
 1   user_id             50000 non-null  object
 2   location            50000 non-null  object
 3   action              50000 non-null  object
 4   filename            10000 non-null  object
 5   url                 10000 non-null  object
 6   to_email            10000 non-null  object
 7   from_email          10000 non-null  object
 8   role                50000 non-null  object
 9   department          50000 non-null  object
 10  province            50000 non-null  object
 11  subject_permission  50000 non-null  object
 12  hour                50000 non-null  int64 
 13  day                 50000 non-null  int64 
 14  month               50000 non-null  int64 
 15  weekday             50000 non-null  object
 16  after_hours         50

3. Creating the Insider Threat Labels

In [11]:
activities["insider_threat"] = 0

print(activities["insider_threat"].value_counts())

insider_threat
0    50000
Name: count, dtype: int64


4. Choosing number of suspicous employees

In [14]:
import numpy as np

# Approximately 5% of employees become insiders
suspicious_users = np.random.choice(
    activities["user_id"].unique(),
    size=int(len(activities["user_id"].unique()) * 0.05),
    replace=False
)

activities.loc[
    activities["user_id"].isin(suspicious_users),
    "insider_threat"
] = 1

print("Suspicious Employees:", len(suspicious_users))

print("\nActivity Distribution")
print(activities["insider_threat"].value_counts())

Suspicious Employees: 50

Activity Distribution
insider_threat
0    47830
1     2170
Name: count, dtype: int64


5. Late night access

In [17]:
mask = activities["insider_threat"] == 1

activities.loc[mask, "hour"] = np.random.randint(0, 4, mask.sum())

activities.loc[mask, "after_hours"] = 1

print("Late-night access injected.")

Late-night access injected.


6. Many Record Access

In [20]:
activities.loc[
    mask,
    "records_accessed"
] = np.random.randint(500, 1001, mask.sum())

print("Large record access simulated.")

Large record access simulated.


7. Activity done close to examinations

In [23]:
activities.loc[
    mask,
    "exam_period"
] = 1

activities.loc[
    mask,
    "days_to_exam"
] = np.random.randint(0, 5, mask.sum())

print("Exam-period behaviour simulated.")

Exam-period behaviour simulated.


8. Weekend Access

In [26]:
weekend_days = ["Saturday", "Sunday"]

activities.loc[
    mask,
    "weekday"
] = np.random.choice(weekend_days, mask.sum())

activities.loc[
    mask,
    "weekend"
] = 1

print("Weekend activity simulated.")

Weekend activity simulated.


In [28]:
print(activities["insider_threat"].value_counts())

activities.head()

insider_threat
0    47830
1     2170
Name: count, dtype: int64


,timestamp,user_id,location,action,filename,url,to_email,from_email,role,department,...,hour,day,month,weekday,after_hours,weekend,exam_period,records_accessed,days_to_exam,insider_threat
0,2010-01-02 06:49:00,NGF0157,PC-6056,Logon,NaN,NaN,NaN,NaN,Senior Examination Officer,Results Processing,...,6,2,1,Saturday,1,1,0,6,112,0
1,2010-01-02 06:50:00,LRR0148,PC-4275,Logon,NaN,NaN,NaN,NaN,Data Capturer,Logistics,...,6,2,1,Saturday,1,1,0,7,21,0
2,2010-01-02 06:53:04,LRR0148,PC-4124,Logon,NaN,NaN,NaN,NaN,Data Capturer,Logistics,...,6,2,1,Saturday,1,1,0,6,8,0
3,2010-01-02 07:00:00,IRM0931,PC-7188,Logon,NaN,NaN,NaN,NaN,ICT Administrator,Examination Administration,...,7,2,1,Saturday,0,1,0,7,61,0
4,2010-01-02 07:00:00,MOH0273,PC-6699,Logon,NaN,NaN,NaN,NaN,Warehouse Officer,Examination Administration,...,7,2,1,Saturday,0,1,0,7,67,0


9. USB Device Theft

In [31]:
mask = (
    (activities["insider_threat"] == 1)
)

activities.loc[
    mask,
    "action"
] = np.random.choice(
    [
        "USB Insert",
        "File Copy",
        "Download",
        "Email",
        "Logon"
    ],
    size=mask.sum(),
    p=[0.20,0.30,0.15,0.20,0.15]
)

print("Suspicious actions created.")

Suspicious actions created.


8. Confidential Examination Papers

In [34]:
exam_files = [
    "Mathematics_P1_Final.pdf",
    "Mathematics_P2_Final.pdf",
    "Physical_Sciences_P1.pdf",
    "Physical_Sciences_P2.pdf",
    "Accounting_P1.pdf",
    "Accounting_P2.pdf",
    "Life_Sciences_P1.pdf",
    "Life_Sciences_P2.pdf",
    "English_FAL_P1.pdf",
    "English_FAL_P2.pdf"
]

mask = (
    (activities["insider_threat"] == 1) &
    (activities["action"] == "File Copy")
)

activities.loc[
    mask,
    "filename"
] = np.random.choice(
    exam_files,
    size=mask.sum()
)

print("Confidential examination papers assigned.")

Confidential examination papers assigned.


9. Internal Email Leakage

* creating work email for every employee using their user id

In [38]:
activities["work_email"] = (
    activities["user_id"].str.lower() + "@dbe.gov.za"
)

print("DBE work emails created successfully.")

DBE work emails created successfully.


In [40]:
mask = (
    (activities["insider_threat"] == 1) &
    (activities["action"] == "Email")
)

internal_emails = activities["work_email"].unique()

for idx in activities[mask].index:

    sender = activities.at[idx, "work_email"]

    recipients = [email for email in internal_emails if email != sender]

    activities.at[idx, "from_email"] = sender
    activities.at[idx, "to_email"] = np.random.choice(recipients)

print("Realistic internal DBE email sharing simulated.")

Realistic internal DBE email sharing simulated.


10. Simulating Unauthorized Subject Access

In [43]:
# Create a new column that initially matches the employee's authorised subject
activities["accessed_subject"] = activities["subject_permission"]

# List of all examination subjects
subjects = [
    "Mathematics",
    "Mathematical Literacy",
    "Physical Sciences",
    "Life Sciences",
    "Accounting",
    "Business Studies",
    "Geography",
    "English FAL",
    "English HL"
]

# Select only insider threat records
mask = activities["insider_threat"] == 1

# Change only the accessed subject for insider activities
for idx in activities[mask].index:

    authorised_subject = activities.at[idx, "subject_permission"]

    # Create a list of subjects excluding the authorised one
    available_subjects = [
        subject for subject in subjects
        if subject != authorised_subject
    ]

    # Assign an unauthorised subject
    activities.at[idx, "accessed_subject"] = np.random.choice(available_subjects)

print("Unauthorised subject access simulated successfully.")

Unauthorised subject access simulated successfully.


In [45]:
activities.loc[
    activities["insider_threat"] == 1,
    [
        "user_id",
        "role",
        "department",
        "subject_permission",
        "accessed_subject",
        "insider_threat"
    ]
].head(10)

,user_id,role,department,subject_permission,accessed_subject,insider_threat
149,DRR0242,Provincial Examination Coordinator,Question Paper Development,English HL,Life Sciences,1
255,DRR0242,Provincial Examination Coordinator,Question Paper Development,English HL,Accounting,1
279,JTM0223,Printing Technician,Question Paper Development,Geography,Accounting,1
280,JTM0223,Printing Technician,Question Paper Development,Geography,Physical Sciences,1
324,JTM0223,Printing Technician,Question Paper Development,Geography,Physical Sciences,1
325,JTM0223,Printing Technician,Question Paper Development,Geography,English HL,1
342,DMG0358,Provincial Examination Coordinator,Results Processing,Mathematics,English FAL,1
350,DMG0358,Provincial Examination Coordinator,Results Processing,Mathematics,Mathematical Literacy,1
355,MSO0222,Chief Examination Officer,Question Paper Development,Mathematical Literacy,Geography,1
359,MSO0222,Chief Examination Officer,Question Paper Development,Mathematical Literacy,Physical Sciences,1


12. Simulating Acess To Confidential Examination Papers

In [48]:
exam_files = [
    "Mathematics_P1_Final.pdf",
    "Mathematics_P2_Final.pdf",
    "Physical_Sciences_P1_Final.pdf",
    "Physical_Sciences_P2_Final.pdf",
    "Accounting_P1_Final.pdf",
    "Accounting_P2_Final.pdf",
    "Business_Studies_P1_Final.pdf",
    "Business_Studies_P2_Final.pdf",
    "English_FAL_P1_Final.pdf",
    "English_FAL_P2_Final.pdf",
    "English_HL_P1_Final.pdf",
    "English_HL_P2_Final.pdf",
    "Life_Sciences_P1_Final.pdf",
    "Life_Sciences_P2_Final.pdf",
    "Geography_P1_Final.pdf",
    "Geography_P2_Final.pdf"
]

mask = (
    (activities["insider_threat"] == 1) &
    (activities["action"] == "File Copy")
)

activities.loc[
    mask,
    "filename"
] = np.random.choice(
    exam_files,
    size=mask.sum()
)

print("Confidential examination papers assigned successfully.")

Confidential examination papers assigned successfully.


13. Examination access permission

In [50]:
allowed_exam_roles = {
    "Chief Examination Officer": True,
    "Senior Examination Officer": True,
    "Provincial Examination Coordinator": True,
    "Printing Technician": True,
    "ICT Administrator": False,
    "Data Capturer": False,
    "Warehouse Officer": False
}

activities["allowed_exam_access"] = activities["role"].map(
    allowed_exam_roles
)

print("Exam access permissions assigned.")

Exam access permissions assigned.


14. Check who handled an examination paper

In [53]:
print(activities["filename"].unique()[:20])

[nan 'English_HL_P1_Final.pdf' 'Business_Studies_P2_Final.pdf'
 'Business_Studies_P1_Final.pdf' 'Geography_P2_Final.pdf'
 'Accounting_P2_Final.pdf' 'Accounting_P1_Final.pdf'
 'Physical_Sciences_P2_Final.pdf' 'Physical_Sciences_P1_Final.pdf'
 'Mathematics_P2_Final.pdf' 'Mathematics_P1_Final.pdf'
 'English_FAL_P1_Final.pdf' 'Geography_P1_Final.pdf'
 'Life_Sciences_P2_Final.pdf' 'English_HL_P2_Final.pdf'
 'English_FAL_P2_Final.pdf' 'Life_Sciences_P1_Final.pdf' 'EYPC9Y08.doc'
 'N3LTSU3O.pdf' 'D3D3WC9W.doc']


In [55]:
print(activities["filename"].value_counts(dropna=False))

filename
NaN                               39483
Geography_P2_Final.pdf               52
Physical_Sciences_P2_Final.pdf       49
English_FAL_P1_Final.pdf             46
Mathematics_P2_Final.pdf             46
                                  ...  
095XSR6E.doc                          1
JDJPGP19.txt                          1
KW0W1ZPH.txt                          1
ZEEQAX8Y.pdf                          1
72EZZ4RR.doc                          1
Name: count, Length: 9865, dtype: int64


In [79]:
# Remove old columns if they already exist
activities.drop(
    columns=[
        "handled_exam_paper",
        "unauthorised_exam_access"
    ],
    errors="ignore",
    inplace=True
)

In [81]:
exam_files = [
    "Mathematics_P1_Final.pdf",
    "Mathematics_P2_Final.pdf",
    "Physical_Sciences_P1_Final.pdf",
    "Physical_Sciences_P2_Final.pdf",
    "Accounting_P1_Final.pdf",
    "Accounting_P2_Final.pdf",
    "Business_Studies_P1_Final.pdf",
    "Business_Studies_P2_Final.pdf",
    "English_FAL_P1_Final.pdf",
    "English_FAL_P2_Final.pdf",
    "English_HL_P1_Final.pdf",
    "English_HL_P2_Final.pdf",
    "Life_Sciences_P1_Final.pdf",
    "Life_Sciences_P2_Final.pdf",
    "Geography_P1_Final.pdf",
    "Geography_P2_Final.pdf"
]
activities["handled_exam_paper"] = (
    activities["filename"].isin(exam_files)
)

print("Exam paper handling identified.")

Exam paper handling identified.


15.  Flag unauthorized exam paper access

In [85]:
activities["unauthorised_exam_access"] = (
    (activities["handled_exam_paper"] == True) &
    (activities["allowed_exam_access"] == False)
)

print("Unauthorised exam access detected.")

Unauthorised exam access detected.


In [91]:
activities.loc[
    activities["handled_exam_paper"] == True,
    [
        "user_id",
        "role",
        "filename",
        "allowed_exam_access",
        "handled_exam_paper",
        "unauthorised_exam_access"
    ]
].head(20)

,user_id,role,filename,allowed_exam_access,handled_exam_paper,unauthorised_exam_access
149,DRR0242,Provincial Examination Coordinator,English_HL_P1_Final.pdf,True,True,False
350,DMG0358,Provincial Examination Coordinator,Business_Studies_P2_Final.pdf,True,True,False
359,MSO0222,Chief Examination Officer,Business_Studies_P1_Final.pdf,True,True,False
475,GCC0945,Data Capturer,Geography_P2_Final.pdf,False,True,True
535,CJK0200,Printing Technician,Accounting_P2_Final.pdf,True,True,False
590,RAR0725,Chief Examination Officer,Accounting_P1_Final.pdf,True,True,False
602,CWB0065,Chief Examination Officer,Physical_Sciences_P2_Final.pdf,True,True,False
648,CWB0065,Chief Examination Officer,Business_Studies_P1_Final.pdf,True,True,False
695,IKR0401,Data Capturer,Physical_Sciences_P1_Final.pdf,False,True,True
696,JAD0609,Warehouse Officer,Business_Studies_P2_Final.pdf,False,True,True


In [93]:
print(activities["handled_exam_paper"].value_counts())

handled_exam_paper
False    49331
True       669
Name: count, dtype: int64


16. Creating Risk score

In [96]:
activities["risk_score"] = 0

17. After-hours activity (+20)

In [99]:
activities.loc[
    activities["after_hours"] == 1,
    "risk_score"
] += 20

18. Weekend activity (+10)

In [102]:
activities.loc[
    activities["weekend"] == 1,
    "risk_score"
] += 10

In [ ]:
19. High number of records accessed (+20)

In [ ]:
activities.loc[
    activities["records_accessed"] >= 500,
    "risk_score"
] += 20

In [ ]:
20. Unauthorised subject access (+30

In [104]:
activities.loc[
    activities["subject_permission"] != activities["accessed_subject"],
    "risk_score"
] += 30

21. Unauthorised exam paper access (+30)

In [108]:
activities.loc[
    activities["unauthorised_exam_access"] == True,
    "risk_score"
] += 30

22. USB activity (+25)

In [111]:
activities.loc[
    activities["action"] == "USB Insert",
    "risk_score"
] += 25

23. Suspicious Email (+20)

In [114]:
activities.loc[
    (activities["action"] == "Email") &
    (activities["insider_threat"] == 1),
    "risk_score"
] += 20

24. Download activity (+15)

In [ ]:
activities.loc[
    activities["action"] == "Download",
    "risk_score"
] += 15

25. Capping the maximum score at 100

In [118]:
activities["risk_score"] = activities["risk_score"].clip(upper=100)

26. Creating Risk levels

In [124]:
activities["risk_level"] = "Low"

activities.loc[
    activities["risk_score"] >= 40,
    "risk_level"
] = "Medium"

activities.loc[
    activities["risk_score"] >= 70,
    "risk_level"
] = "High"

27. Check the Results

In [127]:
activities[
    [
        "user_id",
        "role",
        "action",
        "risk_score",
        "risk_level",
        "insider_threat"
    ]
].head(20)

,user_id,role,action,risk_score,risk_level,insider_threat
0,NGF0157,Senior Examination Officer,Logon,30,Low,0
1,LRR0148,Data Capturer,Logon,30,Low,0
2,LRR0148,Data Capturer,Logon,30,Low,0
3,IRM0931,ICT Administrator,Logon,10,Low,0
4,MOH0273,Warehouse Officer,Logon,10,Low,0
5,LAP0338,Senior Examination Officer,Logon,10,Low,0
6,MHH0180,Data Capturer,Logon,10,Low,0
7,NOB0181,Provincial Examination Coordinator,Logon,10,Low,0
8,AHC0142,Warehouse Officer,Logon,10,Low,0
9,CTR0341,Chief Examination Officer,Logon,10,Low,0


28. View only High-Risk Employees

In [130]:
activities.loc[
    activities["risk_level"] == "High",
    [
        "user_id",
        "role",
        "action",
        "risk_score",
        "risk_level",
        "insider_threat"
    ]
].head(20)

,user_id,role,action,risk_score,risk_level,insider_threat
255,DRR0242,Provincial Examination Coordinator,Email,80,High,1
324,JTM0223,Printing Technician,USB Insert,85,High,1
342,DMG0358,Provincial Examination Coordinator,Email,80,High,1
355,MSO0222,Chief Examination Officer,USB Insert,85,High,1
446,CWB0065,Chief Examination Officer,Email,80,High,1
475,GCC0945,Data Capturer,File Copy,90,High,1
488,JSH0330,Senior Examination Officer,Email,80,High,1
513,DMG0358,Provincial Examination Coordinator,USB Insert,85,High,1
547,MSO0222,Chief Examination Officer,USB Insert,85,High,1
556,JTM0223,Printing Technician,Email,80,High,1


29. Save the Final Dataset

In [133]:
activities.to_csv(
    "DBE_Insider_Threat_Dataset.csv",
    index=False
)

print("Final DBE Insider Threat Dataset saved successfully!")

Final DBE Insider Threat Dataset saved successfully!
